# 真实再扩散攻击闭环运行入口

该 Notebook 用于 Colab 冷启动: 挂载 Google Drive、拉取仓库、读取前序 aligned rescoring 图像包、加载 SD3.5 Medium image-to-image pipeline、生成真实 attacked image 文件、记录 source / attacked image digest、重跑攻击后检测代理分数, 并把所有核对文件打包到 Google Drive。

正式逻辑位于 `paper_workflow/colab_utils/real_attack_evaluation.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 确认 Hugging Face 账号已获得 SD3.5 Medium 与 DDIM attacker model 访问权限, 并配置 `HF_TOKEN`。
3. 确认 Google Drive 中已有 aligned rescoring 包, 默认查找 `/content/drive/MyDrive/SLM/aligned_rescoring/aligned_rescoring_package_*.zip`。
4. 确认 Google Drive 中已有 threshold calibration 包, 默认查找 `/content/drive/MyDrive/SLM/threshold_calibration/threshold_calibration_package_*.zip`。
5. 默认输出目录为 `/content/drive/MyDrive/SLM/real_attack_evaluation`。
6. 本入口会先打包所有核对文件, 再执行断言, 便于失败时回传诊断结果。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/real_attack_evaluation')
os.environ.setdefault('SLM_WM_ALIGNED_RESCORING_DRIVE_DIR', '/content/drive/MyDrive/SLM/aligned_rescoring')
os.environ.setdefault('SLM_WM_THRESHOLD_CALIBRATION_DRIVE_DIR', '/content/drive/MyDrive/SLM/threshold_calibration')
os.environ.setdefault('SLM_WM_REAL_ATTACK_SOURCE_IMAGE_DIR', 'outputs/aligned_rescoring/aligned_images')
os.environ.setdefault('SLM_WM_REAL_ATTACK_SOURCE_COUNT', '1')
os.environ.setdefault('SLM_WM_REQUIRE_ALL_REGEN_ATTACKS', '1')
os.environ.setdefault('SLM_WM_DDIM_ATTACK_MODEL_ID', 'runwayml/stable-diffusion-v1-5')
print({
    'drive_output_dir': os.environ['SLM_WM_DRIVE_OUTPUT_DIR'],
    'aligned_rescoring_drive_dir': os.environ['SLM_WM_ALIGNED_RESCORING_DRIVE_DIR'],
    'threshold_calibration_drive_dir': os.environ['SLM_WM_THRESHOLD_CALIBRATION_DRIVE_DIR'],
    'source_image_dir': os.environ['SLM_WM_REAL_ATTACK_SOURCE_IMAGE_DIR'],
    'ddim_attack_model_id': os.environ['SLM_WM_DDIM_ATTACK_MODEL_ID'],
})


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub


In [ ]:
import importlib.metadata as importlib_metadata
import importlib.util

import diffusers
import transformers

def package_version_or_missing(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return 'not_installed'

dependency_report = {
    'diffusers': diffusers.__version__,
    'transformers': transformers.__version__,
    'accelerate': package_version_or_missing('accelerate'),
    'huggingface_hub': package_version_or_missing('huggingface_hub'),
    'safetensors': package_version_or_missing('safetensors'),
    'sentencepiece': package_version_or_missing('sentencepiece'),
    'protobuf': package_version_or_missing('protobuf'),
    'stable_diffusion_img2img_available': hasattr(diffusers, 'StableDiffusion3Img2ImgPipeline'),
    'auto_img2img_available': hasattr(diffusers, 'AutoPipelineForImage2Image'),
    'ddim_scheduler_available': hasattr(diffusers, 'DDIMScheduler'),
    'pillow_available': importlib.util.find_spec('PIL') is not None,
}
print(dependency_report)


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
import os

from paper_workflow.colab_utils.real_attack_evaluation import run_default_real_attack_evaluation_from_drive_plan

real_attack_summary = run_default_real_attack_evaluation_from_drive_plan(
    root='.',
    aligned_rescoring_drive_dir=os.environ['SLM_WM_ALIGNED_RESCORING_DRIVE_DIR'],
    threshold_calibration_drive_dir=os.environ['SLM_WM_THRESHOLD_CALIBRATION_DRIVE_DIR'],
    require_threshold_package=True,
)
real_attack_summary


In [ ]:
from pathlib import Path

summary_path = Path('outputs/real_attack_evaluation/real_attack_run_summary.json')
input_manifest_path = Path('outputs/real_attack_evaluation/real_attack_input_package_manifest.json')
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else real_attack_summary)
if input_manifest_path.exists():
    print(input_manifest_path.read_text(encoding='utf-8'))


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.real_attack_evaluation import package_real_attack_evaluation_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ.get('SLM_WM_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/real_attack_evaluation')
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'real_attack_evaluation_package_{utc_suffix}_{short_commit}.zip'
archive_record = package_real_attack_evaluation_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/real_attack_evaluation').glob('*.json')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8'))

for result_path in sorted(Path('outputs/real_attack_evaluation').glob('*.csv')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8')[:4000])

records_path = Path(real_attack_summary.get('output_records_path', ''))
formal_records_path = Path(real_attack_summary.get('formal_records_path', ''))
if records_path.exists():
    print(records_path.read_text(encoding='utf-8')[:4000])
if formal_records_path.exists():
    print(formal_records_path.read_text(encoding='utf-8')[:4000])

assert real_attack_summary['run_decision'] == 'pass', real_attack_summary
assert real_attack_summary['real_attacked_image_closed_loop_ready'] is True, real_attack_summary
assert real_attack_summary['regeneration_attack_gpu_validation_ready'] is True, real_attack_summary
assert real_attack_summary['attack_detection_rerun_ready'] is True, real_attack_summary
assert real_attack_summary['formal_attack_detection_ready'] is True, real_attack_summary
assert real_attack_summary['real_attacked_image_count'] > 0, real_attack_summary
